In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "dbstorageproyecto";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_au
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.FilmDetails (
id integer,
director string,
top_billed string,
budget_usd double,
revenue_usd double
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/FilmDetails"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.Movies (
id integer,
title string,
genres string,
language string,
user_score double,
runtime_hour integer,
runtime_min integer,
release_date timestamp,
vote_count integer
)
USING DELTA
PARTITIONED BY (release_date)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/Movies"

###Tablas Silver

In [0]:
drop table catalog_au.silver.Movies_FilmDetails

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.Movies_FilmDetails (
  movie_id integer,
  title string,
  runtime_min integer,
  user_score double,
  director string,
  movie_title_details string
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/Movies_FilmDetails"


###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.Movies_FilmDetails_partitioned (
  conteo long,
  max_user_score integer,
  min_runtime integer,
  max_vote integer
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_Movies_FilmDetails_partitioned"